**TODO:明天开始从头写，一点点替代PyTorch**

In [1]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST

train_data = MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(batch_size=64, shuffle=True, num_workers=0, dataset=train_data)

In [ ]:
import numpy as np

class ReLU:
    def __init__(self):
        self.x = None
        self.dA_dL = None

    def forward(self, x):
        self.x = x
        return np.maximum(0, x)

    def backward(self, dZ_dW):
        return dZ_dW * (self.x > 0)

class Linear:
    def __init__(self, in_dim, out_dim, lr=0.001):
        self.W = np.random.randn(in_dim, out_dim) * lr
        self.b = np.zeros(out_dim)
        self.lr = lr
        self.A_prev = None
        self.dW = None
        self.db = None

    def forward(self, A_prev):
        self.A_prev = A_prev
        return self.A_prev @ self.W + self.b

    def backward(self, dL_dZ):
        self.dW = self.A_prev.T @ dL_dZ
        self.db = np.sum(dL_dZ, axis=0)
        return dL_dZ @ self.W.T

    def step(self):
        self.W -= self.lr * self.dW
        self.b -= self.lr * self.db

class CrossEntropyLoss:
    def __init__(self):
        self.A_softmax = None
        self.A = None
        self.t = None
        self.N = None

    def forward(self, A, t_onehot):
        N = A.shape[0]
        self.A = A
        A_exp = np.exp(self.A - np.max(self.A))

        self.A_softmax = A_exp / np.sum(A_exp, axis=1, keepdims=True)
        self.t = t_onehot
        self.N = N

        if self.t.ndim == 1:
            t_onehot = np.zeros_like(self.A_softmax)
            t_onehot[np.arange(self.N), t_onehot] = 1
            self.t = t_onehot
        else:
            self.t = t_onehot

    def backward(self):
        return (self.A_softmax - self.t) / self.N

